In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as datasets
from torchvision import transforms

from IPython.display import clear_output
from tqdm.notebook import tqdm as tqdm
import ot
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Data

In [ ]:
BATCH_SIZE = 16

def random_color(im):
    hue = 360*np.random.rand()
    d = (im *(hue%60)/60)
    im_min, im_inc, im_dec = torch.zeros_like(im), d, im - d
    c_im = torch.zeros((3, im.shape[1], im.shape[2]))
    H = round(hue/60) % 6
    cmap = [[0, 3, 2], [2, 0, 3], [1, 0, 3], [1, 2, 0], [3, 1, 0], [0, 1, 2]]
    return torch.cat((im, im_min, im_dec, im_inc), dim=0)[cmap[H]]

TRANSFORM = transforms.Compose([
    transforms.Resize(16),
    transforms.ToTensor(),
    random_color,
    transforms.Normalize([0.5],[0.5])
])

# Load train datasets
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=TRANSFORM)
idx = np.array(range(len(mnist_train)))
mnist_2 = torch.utils.data.Subset(mnist_train, idx[mnist_train.targets==2])
mnist_3 = torch.utils.data.Subset(mnist_train, idx[mnist_train.targets==3])
mnist_2_loader = torch.utils.data.DataLoader(mnist_2, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
mnist_3_loader = torch.utils.data.DataLoader(mnist_3, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# Load test datasets
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=TRANSFORM)
idx = np.array(range(len(mnist_test)))
mnist_2_test = torch.utils.data.Subset(mnist_test, idx[mnist_test.targets==2])
mnist_3_test = torch.utils.data.Subset(mnist_test, idx[mnist_test.targets==3])
mnist_2_test_loader = torch.utils.data.DataLoader(mnist_2_test, batch_size=BATCH_SIZE)
mnist_3_test_loader = torch.utils.data.DataLoader(mnist_3_test, batch_size=BATCH_SIZE)

# We pick a few samples from them for the qualitative analysis
X_test_fixed = next(iter(mnist_2_test_loader))[0]
Y_test_fixed = next(iter(mnist_3_test_loader))[0]


In [ ]:
iter_mnist_2, iter_mnist_3 = iter(mnist_2_loader), iter(mnist_3_loader)

def sample_mnist_2():
    global iter_mnist_2, mnist_2_loader
    try:
        return next(iter_mnist_2)[0]
    except StopIteration:
        iter_mnist_2 = iter(mnist_2_loader)
        return next(iter_mnist_2)[0]

def sample_mnist_3():
    global iter_mnist_3, mnist_3_loader
    try:
        return next(iter_mnist_3)[0]
    except StopIteration:
        iter_mnist_3 = iter(mnist_3_loader)
        return next(iter_mnist_3)[0]

In [ ]:
def plot_images(batch):
    batch = batch.cpu()
    fig, axes = plt.subplots(1, 10, figsize=(10, 1), dpi=100)
    for i in range(10):
        axes[i].imshow(batch[i].mul(0.5).add(0.5).clip(0,1).permute(1,2,0))
        axes[i].set_xticks([]); axes[i].set_yticks([])
    fig.tight_layout(pad=0.1)

print('Random (unpaired) images from Colored MNIST-2 (1st row) and Colored MNIST-3 (2nd row) train sets')
plot_images(sample_mnist_2())
plot_images(sample_mnist_3())

## Architectures

In [ ]:
T = nn.Sequential(
    nn.Conv2d(3, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.Conv2d(128, 3, kernel_size=5, padding=2),
).to(DEVICE)

f = nn.Sequential(
    nn.Conv2d(3, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  128 x 8 x 8
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  256 x 4 x 4
    nn.Conv2d(128, 128, kernel_size=5, padding=2),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  512 x 2 x 2
    nn.Conv2d(128, 128, kernel_size=3, padding=1),
    nn.ReLU(True),
    nn.AvgPool2d(2), #  512 x 1 x 1
    nn.Conv2d(128, 1, kernel_size=1, padding=0),
    nn.Flatten(1),
).to(DEVICE)


print('T params:', np.sum([np.prod(p.shape) for p in T.parameters()]))
print('f params:', np.sum([np.prod(p.shape) for p in f.parameters()]))

def cost(X, T_X):
    X_flat = X.flatten(start_dim=1)
    T_X_flat = T_X.flatten(start_dim=1)

    cost = torch.pow(X_flat - T_X_flat, 2).sum(dim=1).mean()

    return cost


MAX_STEPS = 50000 + 1
T_LR, f_LR = 5e-4, 5e-4
inner_iter = 5
T_opt = torch.optim.Adam(T.parameters(), lr=T_LR, weight_decay=1e-10)
f_opt = torch.optim.Adam(f.parameters(), lr=f_LR, weight_decay=1e-10, maximize = True)

## TTS-EG (Both Semi-Dual Proablem and Lagrangian-based Problem)

In [ ]:
for step in tqdm(range(MAX_STEPS)):

    T_old = [p.clone() for p in T.parameters()]
    f_old = [p.clone() for p in f.parameters()]


    X = sample_mnist_2().to(DEVICE)
    Y = sample_mnist_3().to(DEVICE) 
    
    T_X = T(X) 

    # Strong Cost 사용 (Z가 없으므로 GAMMA 인자는 무시되거나 제거 가능)
    loss = cost(X, T_X).mean() - f(T_X).mean() + f(Y).mean()
    
    T_opt.zero_grad(); f_opt.zero_grad()
    loss.backward()
    T_opt.step(); f_opt.step()


    X = sample_mnist_2().to(DEVICE)
    Y = sample_mnist_3().to(DEVICE)
    
    T_X2 = T(X)
    loss2 = cost(X, T_X2).mean() - f(T_X2).mean() + f(Y).mean()

    T_opt.zero_grad(); f_opt.zero_grad()
    loss2.backward()


    with torch.no_grad():
        for p, p_old in zip(T.parameters(), T_old):
            p.copy_(p_old)
        for p, p_old in zip(f.parameters(), f_old):
            p.copy_(p_old)

    T_opt.step(); f_opt.step()


    if step % 500 == 0:
        clear_output(wait=True)
        print(f"Step {step} | Loss: {loss.item():.6f}")
        
        with torch.no_grad():
            T.eval()
            T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
            T.train()
            
        print('X | T(X) | Y')
        plot_images(X_test_fixed)
        plot_images(T_X_test)
        plot_images(Y_test_fixed)
        plt.show()


if f_LR < T_LR:
    name = "Semi-Dual"
elif f_LR > T_LR:
    name = "Lagrangian"
else:
    name = "Standard"

file_name = name + "_TTS-EG" 

mode_name = "color_to_color"

os.makedirs(mode_name, exist_ok=True)


save_path = os.path.join(mode_name, file_name)

T.eval()
torch.save(T.state_dict(), save_path)

print(f"✅ [{mode_name}] model is saved")

## GDmax (Semi-Dual Problem)

In [ ]:
for step in tqdm(range(MAX_STEPS)):
    T.train()
    f.train()

    X = sample_mnist_2().to(DEVICE)
    Y = sample_mnist_3().to(DEVICE)
    
    with torch.no_grad():
        T_X_f = T(X)

    loss_f = - f(T_X_f).mean() + f(Y).mean()
    
    f_opt.zero_grad()
    loss_f.backward()
    f_opt.step()

    for _ in range(inner_iter):
        X = sample_mnist_2().to(DEVICE)
        
        T_X = T(X)
 
        loss_T = cost(X, T_X).mean() - f(T_X).mean()
        
        T_opt.zero_grad()
        loss_T.backward()
        T_opt.step()

    if step % 500 == 0:
        clear_output(wait=True)
        print(f"Step {step} | Loss: {loss_T.item():.6f}")
        
        with torch.no_grad():
            T.eval()
            T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
            T.train()
            
        print('X | T(X) | Y')
        plot_images(X_test_fixed)
        plot_images(T_X_test)
        plot_images(Y_test_fixed)
        plt.show()

file_name = "Semi-Dual_GDmax"

mode_name = "color_to_color"

os.makedirs(mode_name, exist_ok=True)


save_path = os.path.join(mode_name, file_name)

T.eval()
torch.save(T.state_dict(), save_path)

print(f"✅ [{mode_name}] model is saved")

## GDmax (Lagrangian-Based Problem)

In [ ]:
for step in tqdm(range(MAX_STEPS)):
    T.train()
    f.train()
    X = sample_mnist_2().to(DEVICE)
        
    T_X = T(X)

    loss_T = cost(X, T_X).mean() - f(T_X).mean()
    
    T_opt.zero_grad()
    loss_T.backward()
    T_opt.step()
    

    for _ in range(inner_iter):
        X = sample_mnist_2().to(DEVICE)
        Y = sample_mnist_3().to(DEVICE)
        
        with torch.no_grad():
            T_X_f = T(X)

        loss_f = - f(T_X_f).mean() + f(Y).mean()
        
        f_opt.zero_grad()
        loss_f.backward()
        f_opt.step()

    if step % 500 == 0:
        clear_output(wait=True)
        print(f"Step {step} | Loss: {loss_T.item():.6f}")
        
        with torch.no_grad():
            T.eval()
            T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
            T.train()
            
        print('X | T(X) | Y')
        plot_images(X_test_fixed)
        plot_images(T_X_test)
        plot_images(Y_test_fixed)
        plt.show()


file_name = "Lagrangian_GDmax" # "Lagrangian_EG" 

mode_name = "color_to_color"

os.makedirs(mode_name, exist_ok=True)


save_path = os.path.join(mode_name, file_name)

T.eval()
torch.save(T.state_dict(), save_path)

print(f"✅ [{mode_name}] model is saved")

## Evaluation

In [ ]:
all_X_test, all_Y_test = [], []
for imgs, _ in mnist_2_test_loader:
    all_X_test.append(imgs)
for imgs, _ in mnist_3_test_loader:
    all_Y_test.append(imgs)

X_test_all = torch.cat(all_X_test, dim=0).to(DEVICE)
Y_test_all = torch.cat(all_Y_test, dim=0).to(DEVICE)


T.eval()
with torch.no_grad():
    TX_test_all = torch.clamp(T(X_test_all), -1.0, 1.0)


TX_flat = TX_test_all.view(TX_test_all.size(0), -1).cpu().numpy()
X_flat = X_test_all.view(X_test_all.size(0), -1).cpu().numpy()
Y_flat = Y_test_all.view(Y_test_all.size(0), -1).cpu().numpy()

n, m = X_flat.shape[0], Y_flat.shape[0]
a, b = np.ones((n,)) / n, np.ones((m,)) / m
pixel_dim = 3 * 16 * 16 # 768



M_tx_y = ot.dist(TX_flat, Y_flat, metric='sqeuclidean')
w2_tx_y = ot.emd2(a, b, M_tx_y)
per_pixel_w2_tx_y = w2_tx_y / pixel_dim


M_x_y = ot.dist(X_flat, Y_flat, metric='sqeuclidean')
w2_x_y = ot.emd2(a, b, M_x_y)
per_pixel_w2_x_y = w2_x_y / pixel_dim
per_pixel_transport_cost = np.mean(np.sum((X_flat - TX_flat)**2, axis=1)) / pixel_dim
per_pixel_diff = np.abs(per_pixel_w2_x_y - per_pixel_transport_cost)




print("-" * 65)
print(f"{'Metric Item (Per-Pixel)':<40} | {'Value':<15}")
print("-" * 65)
print(f"{'1. D_target':<40} | {per_pixel_w2_tx_y:.6f}")
print(f"{'2. D_cost':<40} | {per_pixel_diff:.6f}")
print("-" * 65)